In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/544 Project/Train"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
import torch
from torchinfo import summary
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

In [ ]:
df = pd.read_excel("truthfulqa_generation_data.xlsx")
df.head()

In [ ]:
df.size

In [ ]:
def format_dataset(df):
    examples = []
    for _, row in df.iterrows():
        question = row["Question"]
        examples.append({
            "question": question,
            "answer": row["Best Answer"],
            "label": "correct"
        })
        for ans in row["Correct Answers"].split(";"):
            ans = ans.strip()
            if ans and ans != row["Best Answer"]:
                examples.append({
                    "question": question,
                    "answer": ans,
                    "label": "correct"
                })
    return pd.DataFrame(examples)

In [ ]:
formatted_df = format_dataset(df)
formatted_df.head()

In [ ]:
formatted_df.size

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

In [ ]:
def apply_chat_template(row):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful and truthful assistant. Answer questions accurately and honestly. If you are uncertain, say so."
        },
        {
            "role": "user",
            "content": row["question"]
        },
        {
            "role": "assistant",
            "content": row["answer"]
        }
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

In [ ]:
hf_dataset = Dataset.from_pandas(formatted_df[["question", "answer"]])
hf_dataset = hf_dataset.map(apply_chat_template)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
summary(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
summary(model)

In [ ]:
response_template = "<|im_start|>assistant\n"

In [ ]:
# collator = DataCollatorForCompletionOnlyLM(
#     response_template=response_template,
#     tokenizer=tokenizer
# )

In [ ]:
training_args = SFTConfig(
    output_dir="./",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    save_strategy="epoch",
    warmup_steps=0.05,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    report_to="none",
    max_length=512,
    dataset_text_field="text",
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=hf_dataset,
    eval_dataset=hf_dataset,
    args=training_args,
)

In [ ]:
pre_eval = trainer.evaluate()
print("Baseline Qwen2.5-3B Model: ")
for k, v in pre_eval.items():
    print(f"  {k}: {v}")

In [ ]:
trainer.train()

In [ ]:
post_eval = trainer.evaluate()
print("LoRA Fine-tuned Qwen2.5-3B Model: ")
for k, v in post_eval.items():
    print(f"  {k}: {v}")

In [ ]:
print("Improvements: ")
for k in post_eval:
    if k in pre_eval and isinstance(post_eval[k], float):
        delta = post_eval[k] - pre_eval[k]
        print(f"  {k}: {'+' if delta > 0 else ''}{delta:.4f}")

In [ ]:
model.save_pretrained("./Qwen2.5-3B/LoRA/model")
tokenizer.save_pretrained("./Qwen2.5-3B/LoRA/tokenizer")